# 28 — Locked 14-class creator benchmark evidence

**Runtime: CPU (high-RAM CPU if available).** T4 does not materially speed up
MediaPipe pose extraction in this notebook.

This notebook prepares independent evaluation evidence. It does **not** train,
calibrate, select a threshold, or run a model. `Runtime → Run all` automatically:

1. downloads the pinned source manifest from GitHub and verifies SHA-256;
2. rejects every known training/evaluation video ID and creator;
3. downloads the four temporary sources at 360p with retry/fallback logic;
4. records safe metadata and compact full-timeline visual contact sheets;
5. extracts continuous 17-joint landmarks at 15 FPS;
6. downloads one ZIP named `calorie_creator_benchmark_v1_evidence.zip`.

Raw videos are temporary and are never included in the ZIP. Projected class
coverage is only a locator. Labels remain locked until visible-motion review of
the returned evidence is complete.


In [ ]:
%pip install -q --upgrade "yt-dlp[default]" mediapipe==0.10.35 opencv-python-headless pandas==2.2.3 tqdm


In [ ]:
from pathlib import Path
import hashlib
import json
import math
import os
import re
import shutil
import subprocess
import urllib.request

import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
from google.colab import files
from IPython.display import display
from tqdm.auto import tqdm

PINNED_DATA_COMMIT = '062f43d86208f06930ae87297c421abe82e164b9'
RAW_BASE = f'https://raw.githubusercontent.com/bariscelikk1/cALorie/{PINNED_DATA_COMMIT}'
TARGET_FPS = 15.0
WORK_ROOT = Path('/content/calorie_creator_benchmark_v1_work')
VIDEO_ROOT = WORK_ROOT / 'videos'
OUTPUT_ROOT = Path('/content/calorie_creator_benchmark_v1_evidence')
LANDMARK_ROOT = OUTPUT_ROOT / 'continuous_landmarks'
CONTACT_ROOT = OUTPUT_ROOT / 'timeline_contact_sheets'
METADATA_ROOT = OUTPUT_ROOT / 'source_metadata'
for directory in (WORK_ROOT, VIDEO_ROOT, OUTPUT_ROOT, LANDMARK_ROOT, CONTACT_ROOT, METADATA_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

DENO_INSTALL = WORK_ROOT / 'deno_runtime'
DENO_BIN = DENO_INSTALL / 'bin' / 'deno'
if not DENO_BIN.exists():
    install_env = os.environ.copy()
    install_env['DENO_INSTALL'] = str(DENO_INSTALL)
    subprocess.run(
        ['bash', '-lc', 'curl -fsSL https://deno.land/install.sh | sh'],
        env=install_env, check=True,
    )
deno_version = subprocess.run(
    [str(DENO_BIN), '--version'], text=True, stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT, check=True,
).stdout.splitlines()[0]
YTDLP_RUNTIME_ARGS = [
    '--js-runtimes', f'deno:{DENO_BIN}',
    '--remote-components', 'ejs:npm',
]

EXPECTED_SHA256 = {
    'candidates.csv': '51285f59454bf2fc3966d93ee9abef37775dcd0b098c1d75895ac55f95d82d7a',
    'projected_coverage_matrix.csv': '5565fdca97e08c5956422714450bcd41a0fd72ccbe0c4c4430ce4b9a982d3a7c',
    'excluded_source_inventory.csv': 'ecf18c29ef5b7ba1eca3f45677e463df256b965888f3913ae55606807baf2232',
}
print('Runtime: CPU')
print('JavaScript runtime:', deno_version)
print('Pinned source commit:', PINNED_DATA_COMMIT)
print('Final output:', OUTPUT_ROOT.with_suffix('.zip'))


## Fetch and enforce the locked benchmark contract

The source list is immutable for this run. Existing YouTube sources and
creators are rejected before any video download begins.


In [ ]:
def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as source:
        for block in iter(lambda: source.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def pinned_download(relative_url, destination, expected_sha):
    destination = Path(destination)
    if not destination.exists() or sha256(destination) != expected_sha:
        temporary = destination.with_suffix(destination.suffix + '.part')
        temporary.unlink(missing_ok=True)
        urllib.request.urlretrieve(f'{RAW_BASE}/{relative_url}', temporary)
        assert sha256(temporary) == expected_sha, f'Checksum mismatch: {destination.name}'
        temporary.replace(destination)
    assert sha256(destination) == expected_sha
    return destination


local_inputs = {}
for name, digest in EXPECTED_SHA256.items():
    local_inputs[name] = pinned_download(
        f'ml/data/youtube_creator_benchmark_v1/{name}', WORK_ROOT / name, digest
    )

candidates = pd.read_csv(local_inputs['candidates.csv'])
coverage = pd.read_csv(local_inputs['projected_coverage_matrix.csv'])
excluded = pd.read_csv(local_inputs['excluded_source_inventory.csv'])
bool_columns = ['training_allowed', 'calibration_allowed', 'threshold_selection_allowed']
for column in bool_columns:
    parsed = candidates[column].map(
        lambda value: value if isinstance(value, (bool, np.bool_))
        else str(value).strip().casefold() in {'1', 'true', 'yes'}
    )
    assert not parsed.any(), f'{column} must remain false'
assert candidates.video_id.is_unique and candidates.creator.is_unique
assert set(candidates.video_id).isdisjoint(set(excluded.video_id))
assert set(candidates.creator.str.casefold()).isdisjoint(set(excluded.creator.str.casefold()))
assert len(coverage) == 14 and (coverage.projected_creator_count >= 1).all()
assert (candidates.benchmark_role == 'locked_external_test').all()
print(f'Locked sources: {len(candidates)} videos / {candidates.creator.nunique()} creators')
print(f'Approximate source duration: {candidates.approx_duration_s.sum()/60:.1f} minutes')
display(coverage[['canonical_class', 'projected_creator_count', 'target_status']])


## Capture metadata and download 360p video

Each attempt prints the end of `yt-dlp` output if it fails. Completed valid
downloads are reused after runtime-cell reruns. The decoded height is checked
so a larger stream is not processed accidentally.


In [ ]:
VIDEO_SUFFIXES = {'.mp4', '.mkv', '.webm', '.mov'}
DOWNLOAD_ATTEMPTS = [
    {
        'name': '360p combined MP4',
        'format': '18/b[height=360][ext=mp4]/b[height=360]',
        'extra': [],
    },
    {
        'name': '360p web-embedded client',
        'format': '18/b[height=360]/bv*[height=360]+ba',
        'extra': ['--extractor-args', 'youtube:player_client=web_embedded'],
    },
    {
        'name': '360p Android VR client',
        'format': '18/b[height=360]/bv*[height=360]+ba',
        'extra': ['--extractor-args', 'youtube:player_client=android_vr'],
    },
    {
        'name': '360p web-safari HLS client',
        'format': 'b[height=360][protocol^=m3u8]/bv*[height=360][protocol^=m3u8]+ba',
        'extra': ['--extractor-args', 'youtube:player_client=web_safari'],
    },
    {
        'name': '360p generic final fallback',
        'format': 'bv*[height=360]+ba/b[height=360]',
        'extra': [],
    },
]


def normalize_name(value):
    return re.sub(r'[^a-z0-9]+', '', str(value).casefold())


def probe_video(path):
    cap = cv2.VideoCapture(str(path))
    values = {
        'fps': float(cap.get(cv2.CAP_PROP_FPS) or 0),
        'frame_count': int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0),
        'width': int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 0),
        'height': int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0),
    }
    cap.release()
    values['duration_s'] = values['frame_count'] / values['fps'] if values['fps'] else 0
    return values


def video_matches(video_id):
    return sorted(
        path for path in VIDEO_ROOT.glob(f'{video_id}.*')
        if path.suffix.lower() in VIDEO_SUFFIXES and path.stat().st_size > 1_000_000
    )


def safe_metadata(video_id):
    url = f'https://www.youtube.com/watch?v={video_id}'
    command = [
        'yt-dlp', '--no-playlist', '--skip-download', '--dump-single-json',
        '--no-warnings', *YTDLP_RUNTIME_ARGS, url,
    ]
    result = subprocess.run(command, text=True, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, check=False)
    if result.returncode:
        raise RuntimeError(f'Metadata failed for {video_id}:\n' + '\n'.join(result.stdout.splitlines()[-30:]))
    raw = json.loads(result.stdout)
    keep = [
        'id', 'title', 'channel', 'channel_id', 'uploader', 'uploader_id',
        'duration', 'upload_date', 'availability', 'license', 'webpage_url',
        'width', 'height', 'fps', 'chapters',
    ]
    return {key: raw.get(key) for key in keep}


def download_360p(row):
    existing = video_matches(row.video_id)
    if len(existing) == 1 and probe_video(existing[0])['height'] == 360:
        print('Reusing:', existing[0].name)
        return existing[0]
    for path in existing:
        path.unlink(missing_ok=True)

    url = row.source_url
    output_template = VIDEO_ROOT / f'{row.video_id}.%(ext)s'
    failures = []
    for attempt in DOWNLOAD_ATTEMPTS:
        for partial in VIDEO_ROOT.glob(f'{row.video_id}*.part*'):
            partial.unlink(missing_ok=True)
        command = [
            'yt-dlp', '--no-playlist', '--no-progress', '--retries', '10',
            '--fragment-retries', '10', '--retry-sleep', 'fragment:exp=1:8',
            '--socket-timeout', '30', '-f', attempt['format'],
            '--merge-output-format', 'mp4', '-o', str(output_template),
            *YTDLP_RUNTIME_ARGS, *attempt['extra'], url,
        ]
        print(f'Downloading {row.video_id}: {attempt["name"]}')
        result = subprocess.run(command, text=True, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, check=False)
        matches = video_matches(row.video_id)
        if result.returncode == 0 and len(matches) == 1:
            info = probe_video(matches[0])
            if info['height'] == 360 and info['frame_count'] > 0:
                return matches[0]
            matches[0].unlink(missing_ok=True)
            failures.append(f'{attempt["name"]}: invalid decoded video {info}')
        else:
            tail = '\n'.join(result.stdout.splitlines()[-30:])
            failures.append(f'{attempt["name"]} (exit {result.returncode}):\n{tail}')
            print(tail)
    raise RuntimeError(f'All 360p attempts failed for {row.video_id}:\n\n' + '\n\n'.join(failures))


sources = []
excluded_ids = set(excluded.video_id.astype(str))
excluded_creators = {normalize_name(value) for value in excluded.creator}
for row in candidates.itertuples(index=False):
    metadata = safe_metadata(row.video_id)
    assert metadata['id'] == row.video_id
    observed_names = {normalize_name(metadata.get('channel')), normalize_name(metadata.get('uploader'))}
    assert not (observed_names & excluded_creators), (row.video_id, observed_names & excluded_creators)
    assert row.video_id not in excluded_ids
    metadata_path = METADATA_ROOT / f'{row.video_id}.json'
    metadata_path.write_text(json.dumps(metadata, indent=2, ensure_ascii=False) + '\n')
    video_path = download_360p(row)
    probe = probe_video(video_path)
    assert probe['height'] == 360
    assert probe['duration_s'] >= 120, (row.video_id, probe)
    sources.append({'row': row, 'metadata': metadata, 'video_path': video_path, 'probe': probe})
    print(row.video_id, metadata.get('channel'), f'{probe["width"]}x{probe["height"]}',
          f'{probe["duration_s"]/60:.1f} min')


## Create compact full-timeline visual evidence

Long videos are sampled every 10 seconds and short demonstration videos every
3 seconds. Twelve timestamped frames are packed into each JPEG. These sheets
locate visible motion for later independent labeling; they do not approve a
class by themselves.


In [ ]:
def read_frame(cap, time_s):
    cap.set(cv2.CAP_PROP_POS_MSEC, max(0.0, float(time_s)) * 1000.0)
    ok, frame = cap.read()
    if not ok:
        raise RuntimeError(f'Could not decode frame at {time_s:.2f}s')
    return frame


def thumbnail(frame, time_s, width=320, height=180):
    frame_h, frame_w = frame.shape[:2]
    scale = min(width / frame_w, height / frame_h)
    resized = cv2.resize(frame, (max(1, round(frame_w * scale)), max(1, round(frame_h * scale))))
    canvas = np.zeros((height, width, 3), dtype=np.uint8)
    y = (height - resized.shape[0]) // 2
    x = (width - resized.shape[1]) // 2
    canvas[y:y+resized.shape[0], x:x+resized.shape[1]] = resized
    label = f'{int(time_s)//60:02d}:{int(time_s)%60:02d}'
    cv2.rectangle(canvas, (0, 0), (82, 26), (0, 0, 0), -1)
    cv2.putText(canvas, label, (6, 19), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 255, 255), 1, cv2.LINE_AA)
    return canvas


contact_rows = []
for source in sources:
    video_id = source['row'].video_id
    duration_s = source['probe']['duration_s']
    interval_s = 3.0 if duration_s <= 900 else 10.0
    sample_times = np.arange(0.5, max(0.6, duration_s - 0.5), interval_s)
    source_root = CONTACT_ROOT / video_id
    source_root.mkdir(parents=True, exist_ok=True)
    cap = cv2.VideoCapture(str(source['video_path']))
    tiles = []
    page_index = 0
    for sample_index, time_s in enumerate(tqdm(sample_times, desc=f'Contact sheets {video_id}')):
        tiles.append(thumbnail(read_frame(cap, time_s), time_s))
        if len(tiles) == 12 or sample_index == len(sample_times) - 1:
            while len(tiles) < 12:
                tiles.append(np.zeros_like(tiles[0]))
            page = np.concatenate([
                np.concatenate(tiles[row*4:(row+1)*4], axis=1) for row in range(3)
            ], axis=0)
            output_path = source_root / f'{page_index:03d}.jpg'
            cv2.imwrite(str(output_path), page, [cv2.IMWRITE_JPEG_QUALITY, 88])
            contact_rows.append({
                'video_id': video_id, 'page_index': page_index,
                'sample_interval_s': interval_s,
                'first_time_s': float(sample_times[max(0, sample_index - 11)]),
                'last_time_s': float(time_s),
                'path': str(output_path.relative_to(OUTPUT_ROOT)),
            })
            tiles = []
            page_index += 1
    cap.release()

contact_index = pd.DataFrame(contact_rows)
contact_index.to_csv(OUTPUT_ROOT / 'contact_sheet_index.csv', index=False)
display(contact_index.groupby('video_id').agg(pages=('page_index', 'count'), interval_s=('sample_interval_s', 'first')))


## Extract continuous 15 FPS pose landmarks

This is the expensive CPU step. The archive keeps normalized x/y/z plus
visibility for the same 17 joints used by the existing pipeline, along with
timestamps and direct-detection masks.


In [ ]:
MODEL_PATH = WORK_ROOT / 'pose_landmarker_full.task'
MODEL_URL = (
    'https://storage.googleapis.com/mediapipe-models/pose_landmarker/'
    'pose_landmarker_full/float16/latest/pose_landmarker_full.task'
)
if not MODEL_PATH.exists():
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
print('Pose model MB:', round(MODEL_PATH.stat().st_size / 1e6, 2))

KEEP_17 = [0, 2, 5, 7, 8, 11, 12, 13, 14, 15, 16, 23, 24, 25, 26, 27, 28]
LEFT_SHOULDER, RIGHT_SHOULDER = 5, 6
LEFT_HIP, RIGHT_HIP = 11, 12


def normalize_pose(frame):
    xyz = frame[:, :3].copy()
    hip_center = (xyz[LEFT_HIP] + xyz[RIGHT_HIP]) / 2.0
    shoulder_center = (xyz[LEFT_SHOULDER] + xyz[RIGHT_SHOULDER]) / 2.0
    scale = max(float(np.linalg.norm(xyz[LEFT_SHOULDER] - xyz[RIGHT_SHOULDER])),
                float(np.linalg.norm(shoulder_center - hip_center)), 1e-6)
    return np.concatenate([(xyz - hip_center) / scale, frame[:, 3:4]], axis=1)


def fill_missing(sequence):
    shape = sequence.shape
    flat = sequence.reshape(shape[0], -1)
    filled = pd.DataFrame(flat).interpolate(limit_direction='both').fillna(0).to_numpy()
    return filled.reshape(shape).astype(np.float32)


def extract_video(video_path):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f'Could not open {video_path}')
    source_fps = float(cap.get(cv2.CAP_PROP_FPS) or 30.0)
    source_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    next_sample, frame_index = 0.0, 0
    poses, timestamps, detected = [], [], []
    options = mp.tasks.vision.PoseLandmarkerOptions(
        base_options=mp.tasks.BaseOptions(model_asset_path=str(MODEL_PATH)),
        running_mode=mp.tasks.vision.RunningMode.VIDEO, num_poses=1,
        min_pose_detection_confidence=0.5, min_pose_presence_confidence=0.5,
        min_tracking_confidence=0.5, output_segmentation_masks=False,
    )
    progress = tqdm(total=source_frames, desc=Path(video_path).stem, unit='frames')
    with mp.tasks.vision.PoseLandmarker.create_from_options(options) as landmarker:
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            time_s = frame_index / source_fps
            frame_index += 1
            progress.update(1)
            if time_s + 1e-9 < next_sample:
                continue
            next_sample += 1.0 / TARGET_FPS
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            result = landmarker.detect_for_video(image, int(round(time_s * 1000)))
            if result.pose_landmarks:
                points = result.pose_landmarks[0]
                pose = np.asarray([
                    [points[i].x, points[i].y, points[i].z, float(points[i].visibility or 0.0)]
                    for i in KEEP_17
                ], dtype=np.float32)
                poses.append(normalize_pose(pose))
                detected.append(True)
            else:
                poses.append(np.full((17, 4), np.nan, dtype=np.float32))
                detected.append(False)
            timestamps.append(time_s)
    progress.close()
    cap.release()
    if not poses:
        raise RuntimeError(f'No frames decoded from {video_path}')
    return (fill_missing(np.stack(poses)), np.asarray(timestamps, dtype=np.float32),
            np.asarray(detected, dtype=bool), source_fps)


In [ ]:
quality_rows = []
for source in sources:
    row = source['row']
    output_path = LANDMARK_ROOT / f'{row.video_id}.npz'
    if output_path.exists():
        item = np.load(output_path, allow_pickle=False)
        landmarks = item['landmarks']
        timestamps = item['timestamps_s']
        detected = item['detected']
        source_fps = float(item['source_fps'])
    else:
        landmarks, timestamps, detected, source_fps = extract_video(source['video_path'])
        np.savez_compressed(
            output_path, landmarks=landmarks.astype(np.float32),
            timestamps_s=timestamps.astype(np.float32), detected=detected,
            source_fps=np.float32(source_fps), target_fps=np.float32(TARGET_FPS),
            video_id=row.video_id, group_id=f'youtube:{row.video_id}',
        )
    detection_rate = float(detected.mean())
    quality_rows.append({
        'video_id': row.video_id, 'creator': row.creator,
        'group_id': f'youtube:{row.video_id}',
        'benchmark_role': 'locked_external_test',
        'landmark_path': str(output_path.relative_to(OUTPUT_ROOT)),
        'sampled_frames': len(landmarks), 'duration_s': float(timestamps[-1]),
        'target_fps': TARGET_FPS, 'source_fps': source_fps,
        'detection_rate': detection_rate,
        'status': 'ok' if detection_rate >= 0.80 else 'needs_pose_review',
        'label_status': 'unlabeled_evidence_needs_visible_motion_review',
        'training_allowed': False, 'calibration_allowed': False,
        'threshold_selection_allowed': False,
    })
    print(row.video_id, landmarks.shape, f'detection={detection_rate:.3f}')

quality = pd.DataFrame(quality_rows)
quality.to_csv(OUTPUT_ROOT / 'continuous_landmark_manifest.csv', index=False)
assert len(quality) == len(candidates)
assert quality.sampled_frames.sum() > 100_000, 'Unexpectedly few sampled frames'
display(quality)


## Package evidence and download one ZIP

The archive contains no raw video and no model predictions. Returned evidence
must be reviewed and labeled before a frozen candidate comparison is allowed.


In [ ]:
for name, path in local_inputs.items():
    shutil.copy2(path, OUTPUT_ROOT / name)

run_metadata = {
    'notebook': '28_colab_extract_creator_benchmark_v1.ipynb',
    'runtime': 'CPU',
    'target_fps': TARGET_FPS,
    'pinned_data_commit': PINNED_DATA_COMMIT,
    'benchmark_role': 'locked_external_test',
    'training_allowed': False,
    'calibration_allowed': False,
    'threshold_selection_allowed': False,
    'model_predictions_included': False,
    'source_count': len(sources),
    'creator_count': int(candidates.creator.nunique()),
    'projected_supported_class_count': int(len(coverage)),
    'label_status': 'unlabeled_evidence_needs_visible_motion_review',
    'next_step': 'independent visible-motion timeline review before frozen-model evaluation',
}
(OUTPUT_ROOT / 'run_metadata.json').write_text(json.dumps(run_metadata, indent=2) + '\n')

checksum_lines = []
for path in sorted(p for p in OUTPUT_ROOT.rglob('*') if p.is_file() and p.name != 'SHA256SUMS.txt'):
    checksum_lines.append(f'{sha256(path)}  {path.relative_to(OUTPUT_ROOT)}')
(OUTPUT_ROOT / 'SHA256SUMS.txt').write_text('\n'.join(checksum_lines) + '\n')

assert not list(OUTPUT_ROOT.rglob('*.mp4'))
assert not list(OUTPUT_ROOT.rglob('*.webm'))
assert len(list(LANDMARK_ROOT.glob('*.npz'))) == len(candidates)
assert len(list(METADATA_ROOT.glob('*.json'))) == len(candidates)

zip_path = Path(shutil.make_archive(str(OUTPUT_ROOT), 'zip', root_dir=OUTPUT_ROOT))
print('Created:', zip_path)
print('Size MB:', round(zip_path.stat().st_size / 1e6, 1))
print('SHA-256:', sha256(zip_path))
files.download(str(zip_path))
